In [4]:
from pathlib import Path
import json
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.stats.proportion import proportion_confint

DATASETS = ["CaseReportBench", "PHEE", "DiscourseEE", "MACCROBAT"]
QG_Models = ["qwen3-4b", "qwen3-8b", "gpt-oss-120b", "gpt-5.2-high", "gemini-3.1-pro-high"]
PD_Models = ["qwen3-4b", "qwen3-8b", "gpt-oss-120b", "gpt-5-mini-medium", "gemini-3.1-pro-high"]

OUTPUTS_SC_DIR = "/dartfs/rc/home/j/f006f3j/lab/omar/LoQA/Outputs/sc"
SETUP = "loqa"
METRIC = "complex-match-f1"
SAVE_PATH = "/dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Code/xDev/loqa_scores_2.png"

QG_DISPLAY_NAMES = {
    "qwen3-4b": "Qwen3-4B",
    "qwen3-8b": "Qwen3-8B",
    "gpt-oss-120b": "GPT-OSS-120B",
    "gpt-5.2-high": "GPT-5.2",
    "gemini-3.1-pro-high": "Gemini-3.1-Pro",
}
PD_DISPLAY_NAMES = {
    "qwen3-4b": "Qwen3-4B",
    "qwen3-8b": "Qwen3-8B",
    "gpt-oss-120b": "GPT-OSS-120B",
    "gpt-5-mini-medium": "GPT-5-Mini",
    "gemini-3.1-pro-high": "Gemini-3.1-Pro",
}
Y_AXIS_LABEL = "F1 Score" if METRIC == "complex-match-f1" else "Relaxed Match F1 (%)"


def _f1(p, r):
    """Harmonic mean of precision and recall (as fractions, not %)."""
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0


def _f1_ci(precision_pct, recall_pct, n_pred, n_gt, alpha=0.05):
    """
    95% CI on F1 via Wilson intervals on precision and recall.
    Returns (f1_lo, f1_hi) in percentage points.
    """
    tp_p = round(precision_pct / 100 * n_pred)
    tp_r = round(recall_pct / 100 * n_gt)
    p_lo, p_hi = proportion_confint(tp_p, n_pred, alpha=alpha, method="wilson")
    r_lo, r_hi = proportion_confint(tp_r, n_gt,   alpha=alpha, method="wilson")
    return _f1(p_lo, r_lo) * 100, _f1(p_hi, r_hi) * 100


def load_scores(dataset, setup, qg_model, pd_model):
    base_dir = Path(OUTPUTS_SC_DIR) / dataset
    filename = f"{setup}-{qg_model}-zs-v0-{dataset}-dev-{pd_model}-zs-v0.json"
    path = base_dir / filename
    if not path.exists():
        return None, None, None, None

    with path.open("r") as f:
        data = json.load(f).get("overall", {})

    relaxed_f1 = data.get("relaxed-match-f1")
    complex_f1 = data.get("complex-match-f1")

    prefix = METRIC.replace("-f1", "")
    prec   = data.get(f"{prefix}-precision")
    rec    = data.get(f"{prefix}-recall")
    n_gt   = data.get("ground-truth-count", 0)
    n_pred = data.get("prediction-count", 0)

    ci_lo, ci_hi = None, None
    if prec is not None and rec is not None and n_pred > 0 and n_gt > 0:
        ci_lo, ci_hi = _f1_ci(prec, rec, n_pred, n_gt)

    return relaxed_f1, complex_f1, ci_lo, ci_hi


# ── Load data ──
datasets_dfs = {}
datasets_cis = {}
for dataset in DATASETS:
    rows, ci_rows = [], []
    for qg_model in QG_Models:
        row = {"QG": qg_model}
        ci_row = {"QG": qg_model}
        for pd_model in PD_Models:
            relaxed, complex_, ci_lo, ci_hi = load_scores(dataset, SETUP, qg_model, pd_model)
            row[(pd_model, "relaxed-match-f1")] = relaxed
            row[(pd_model, "complex-match-f1")] = complex_
            ci_row[(pd_model, "ci_lo")] = ci_lo
            ci_row[(pd_model, "ci_hi")] = ci_hi
        rows.append(row)
        ci_rows.append(ci_row)

    df = pd.DataFrame(rows).set_index("QG")
    df.columns = pd.MultiIndex.from_tuples(df.columns, names=["PD_model", "metric"])
    datasets_dfs[dataset] = df

    ci_df = pd.DataFrame(ci_rows).set_index("QG")
    ci_df.columns = pd.MultiIndex.from_tuples(ci_df.columns, names=["PD_model", "bound"])
    datasets_cis[dataset] = ci_df

# ── Style ──
PD_COLOR_MAP = {
    "qwen3-4b": "#0072B2",
    "qwen3-8b": "#E69F00",
    "gpt-oss-120b": "#009E73",
    "gpt-5-mini-medium": "#D55E00",
    "gemini-3.1-pro-high": "#7A7A7A",
}
PD_HATCH_MAP = {
    "qwen3-4b": "",
    "qwen3-8b": "//",
    "gpt-oss-120b": "\\\\",
    "gpt-5-mini-medium": "-",
    "gemini-3.1-pro-high": "..",
}

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 7.5,
})

fig, axes = plt.subplots(2, 2, figsize=(7.16, 5), sharey=True)
axes = axes.flatten()

handles_legend = None
for ax, dataset in zip(axes, DATASETS):
    df_plot = datasets_dfs[dataset].xs(METRIC, axis=1, level=1).astype(float)
    ci_lo_df = datasets_cis[dataset].xs("ci_lo", axis=1, level=1).astype(float)
    ci_hi_df = datasets_cis[dataset].xs("ci_hi", axis=1, level=1).astype(float)

    x = np.arange(len(QG_Models))
    width = 0.8 / len(PD_Models)

    for i, pd_model in enumerate(PD_Models):
        offset = (i - len(PD_Models) / 2 + 0.5) * width
        vals = df_plot[pd_model].values
        lo = ci_lo_df[pd_model].values
        hi = ci_hi_df[pd_model].values
        yerr = np.clip(np.array([np.nan_to_num(vals - lo, nan=0),
                                 np.nan_to_num(hi - vals, nan=0)]), 0, None)

        ax.bar(
            x + offset, vals, width,
            label=PD_DISPLAY_NAMES.get(pd_model, pd_model),
            color=PD_COLOR_MAP[pd_model],
            edgecolor="black", linewidth=0.5,
            hatch=PD_HATCH_MAP[pd_model], alpha=0.95,
        )
        ax.errorbar(
            x + offset, vals, yerr=yerr,
            fmt="none", ecolor="black", elinewidth=0.6,
            capsize=1.5, capthick=0.5,
        )

    if handles_legend is None:
        handles_legend, labels_legend = ax.get_legend_handles_labels()

    ax.set_xticks(x)
    ax.set_xticklabels([QG_DISPLAY_NAMES.get(m, m) for m in QG_Models],
                       rotation=25, ha="right")
    ax.set_ylabel(Y_AXIS_LABEL)
    ax.set_title(dataset, fontweight="bold", fontstyle="italic")
    ax.set_ylim(0, 105)
    ax.grid(axis="y", alpha=0.2, linewidth=0.5)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.legend(
    handles_legend, labels_legend,
    loc="upper center", bbox_to_anchor=(0.5, 1.0),
    ncol=len(PD_Models), frameon=True,
    columnspacing=1.0, handletextpad=0.4,
)

plt.tight_layout(rect=[0, 0.0, 1, 0.93])
fig.savefig(SAVE_PATH, bbox_inches="tight")
print(f"Saved: {SAVE_PATH}")


Saved: /dartfs-hpc/rc/home/j/f006f3j/lab/omar/LoQA/Code/xDev/loqa_scores_2.png
